# Vista batch session — semantic-geometric-decoupling-framework

Drives `experiments/queue.yaml` through `training.runner --profile vista` for **one
interactive OnDemand Jupyter session** (typically a ~2h walltime slot) inside a larger
Vista allocation window (e.g. 22h). Re-launch this same notebook once per session until
either the queue drains or the window runs out — each run is a self-contained,
checklist-gated slice of the same crash-resilient chain the Slurm path
(`slurm/chain_job.sh`) already runs unattended; see the "Training tier quickstart" in the
repo README for the underlying mechanism.

**Prerequisites**
- Launched from an OnDemand Jupyter app started inside the `pytorch-arm64` container/module
  so `training`, `torch`, and CUDA are already on the path — this notebook does not build
  or activate an environment for you.
- `$SCRATCH` and `$WORK` are set by the Vista session (checked below; the profile silently
  falls back to `./data/vista` otherwise, which is almost never what you want on a real
  allocation).
- Kernel's working directory is the repo root (checked below).

**What each session does, in order:** environment check → session-budget accounting →
queue bootstrap/validation → canary preflight (refuses to spend GPU-hours on a broken
metric — see `training/canary.py`) → launch the runner under a walltime-matched
`SIGTERM` timeout → verify the session actually produced fresh checkpoints, not just a
process that started → append a handoff entry so the *next* session (possibly run by
you, tomorrow, with zero memory of this one) knows exactly where things stand.


## Config — review before the first session of the window

In [ ]:
PROFILE_NAME = "vista"
QUEUE_PATH = "experiments/queue.yaml"

# Wall-clock budget for *this* session. Keep a safety buffer below the app's actual
# walltime request so SIGTERM (checkpoint-and-stop) fires before OnDemand SIGKILLs the
# kernel outright — the runner traps SIGTERM and checkpoints; it cannot trap SIGKILL.
SESSION_MINUTES = 110
SAFETY_BUFFER_MINUTES = 10
KILL_AFTER_SECONDS = 90  # grace period between SIGTERM and a forced SIGKILL, as in local_chain.sh

# Total allocation window this session is one slice of. Only used for the human-readable
# budget readout below — it does not talk to the TACC allocation system.
WINDOW_HOURS = 22.0

# Off by default: the static H1 x H3 x H5 grid runs first, unattended-loop planner
# refill second, once the static grid has been proven end-to-end (see
# training/runner.py:_default_evolving_scheduler and slurm/chain_job.sh).
EVOLVE = False

# Minimum acceptable canary IoU and GPU utilization before we trust a session made
# real progress (not just "a process ran").
MIN_CANARY_IOU = 0.30
LOW_GPU_UTIL_WARN_PCT = 20.0

## Checklist 1 — Environment

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd()
checklist = []  # (label, ok: bool, detail: str) accumulated across the whole session


def check(label, ok, detail=""):
    checklist.append((label, ok, detail))
    mark = "PASS" if ok else "FAIL"
    print(f"[{mark}] {label}" + (f" — {detail}" if detail else ""))
    return ok


check("cwd is repo root", (REPO_ROOT / "pyproject.toml").is_file(), str(REPO_ROOT))
if not (REPO_ROOT / "pyproject.toml").is_file():
    raise RuntimeError(
        "Launch this notebook with the repo root as its working directory "
        "(OnDemand Jupyter: set the app's working directory, or %cd there before running)."
    )

sys.path.insert(0, str(REPO_ROOT))

for var in ("SCRATCH", "WORK"):
    check(
        f"${var} is set",
        var in os.environ,
        os.environ.get(var, "<unset — profile falls back to ./data/vista>"),
    )

check(
    "HF_HUB_OFFLINE=1",
    os.environ.get("HF_HUB_OFFLINE") == "1",
    os.environ.get("HF_HUB_OFFLINE", "<unset>"),
)

import torch  # noqa: E402

check(
    "torch sees a CUDA device",
    torch.cuda.is_available(),
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU visible",
)

nvidia_smi = shutil.which("nvidia-smi")
check("nvidia-smi on PATH", nvidia_smi is not None, nvidia_smi or "")

git_status = subprocess.run(
    ["git", "status", "--short"], cwd=REPO_ROOT, capture_output=True, text=True
)
check(
    "git working tree clean", git_status.stdout.strip() == "", git_status.stdout.strip() or "clean"
)

git_head = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT, capture_output=True, text=True
)
print(f"git HEAD: {git_head.stdout.strip()}")

## Checklist 2 — Profile, paths, and queue

In [ ]:
from training.profile import Profile  # noqa: E402
from training.queue import ExperimentQueue, ExperimentStatus  # noqa: E402

profile = Profile.load(PROFILE_NAME)
print(f"profile: {profile.name}  device={profile.device}  vlm={profile.vlm_model}")
for name, path in (
    ("data_root", profile.paths.data_root),
    ("ckpt_root", profile.paths.ckpt_root),
    ("mirror_root", profile.paths.mirror_root),
):
    path.mkdir(parents=True, exist_ok=True)
    print(f"  {name}: {path}")

check(
    "resolved paths are outside the repo (not silently defaulted)",
    all(
        str(p).startswith(("/scratch", "/work")) or "SCRATCH" in os.environ or "WORK" in os.environ
        for p in (profile.paths.data_root, profile.paths.ckpt_root, profile.paths.mirror_root)
    ),
    str(profile.paths.ckpt_root),
)

# mirror_root is the purge-safe destination (stands in for $WORK) — put session
# bookkeeping there so it survives $SCRATCH purges between allocation windows.
SESSION_STATE_PATH = profile.paths.mirror_root / "vista_session_state.json"
SESSION_LOG_PATH = profile.paths.mirror_root / "vista_session_log.md"
GPU_LOG_DIR = profile.paths.mirror_root / "gpu_logs"
GPU_LOG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
queue_path = REPO_ROOT / QUEUE_PATH
if not queue_path.exists():
    print(f"{queue_path} does not exist — bootstrapping base entries + the H1xH3xH5 grid.")
    print("Review the result before the first real session; this is a starting point, not gospel.")
    queue_path.parent.mkdir(parents=True, exist_ok=True)
    from training.queue import ExperimentSpec
    from training.sweeps import QueuePopulator, default_grid

    base = [
        ExperimentSpec(name="datagen-mini", kind="datagen"),
        ExperimentSpec(name="pointnet-modelnet40", kind="pointnet", config={"epochs": 120}),
        ExperimentSpec(
            name="eval-h2-lifting",
            kind="eval",
            config={"detector": "gt", "lifting": "both", "split": "heldout"},
            requires=("datagen-mini",),
        ),
        ExperimentSpec(
            name="eval-sunrgbd-mini",
            kind="eval",
            config={"detector": "gt", "lifting": "both", "source": "sunrgbd", "max_samples": 50},
        ),
    ]
    populator = QueuePopulator(queue_path)
    populator.merge(base)
    populator.merge(default_grid().specs())

queue = ExperimentQueue(queue_path)
snapshot_before = queue.snapshot()

print(f"{'name':<28}{'kind':<10}{'status':<10}{'retries':<8}requires")
for spec in snapshot_before:
    print(
        f"{spec.name:<28}{spec.kind:<10}{spec.status.value:<10}{spec.retries:<8}{','.join(spec.requires)}"
    )

pending_or_running = [
    s for s in snapshot_before if s.status in (ExperimentStatus.PENDING, ExperimentStatus.RUNNING)
]
check(
    "queue has runnable work",
    bool(pending_or_running) or EVOLVE,
    f"{len(pending_or_running)} pending/running of {len(snapshot_before)}",
)

## Checklist 3 — Canary preflight (WS0 guardrail)

Cheap, fast checks that must pass before this session spends any Vista GPU-hours. A prior pilot run drained an unattended queue at good utilization while silently producing scientifically void rows (3D IoU identically 0.000) — the canary is the guardrail against repeating that on Vista time.

In [ ]:
from training.canary import CanarySuite, SystemHalt  # noqa: E402

canary_ok = True
canary_detail = ""
try:
    CanarySuite.default(min_3d_iou=MIN_CANARY_IOU).assert_healthy()
except SystemHalt as exc:
    canary_ok = False
    canary_detail = str(exc)

check("canary preflight", canary_ok, canary_detail)
if not canary_ok:
    raise SystemHalt(
        "Canary preflight failed — refusing to launch the runner. Fix the underlying "
        "regression (see the canary_detail above and training/canary.py) before spending "
        "any more of the Vista window."
    )

## Checklist 4 — Launch this session's runner slice

In [ ]:
session_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
gpu_log_path = GPU_LOG_DIR / f"gpu.{session_id}.csv"
runner_log_path = GPU_LOG_DIR / f"runner.{session_id}.log"

with open(gpu_log_path, "w") as gpu_log_fh:
    gpu_sampler = subprocess.Popen(
        [
            "nvidia-smi",
            "--query-gpu=timestamp,utilization.gpu,memory.used,memory.total",
            "--format=csv,noheader",
            "-l",
            "60",
        ],
        stdout=gpu_log_fh,
        stderr=subprocess.DEVNULL,
    )
    # Popen already dup'd the fd for the child; the `with` block closing our copy on exit is safe.

slice_minutes = SESSION_MINUTES - SAFETY_BUFFER_MINUTES
if slice_minutes <= 0:
    raise ValueError(
        f"SESSION_MINUTES ({SESSION_MINUTES}) must exceed SAFETY_BUFFER_MINUTES ({SAFETY_BUFFER_MINUTES})"
    )

cmd = [
    "timeout",
    "--signal=SIGTERM",
    f"--kill-after={KILL_AFTER_SECONDS}",
    f"{slice_minutes}m",
    sys.executable,
    "-m",
    "training.runner",
    "--profile",
    PROFILE_NAME,
    "--queue",
    QUEUE_PATH,
]
if EVOLVE:
    cmd.append("--evolve")

print(
    f"session {session_id}: launching for up to {slice_minutes} min "
    f"(SIGTERM then {KILL_AFTER_SECONDS}s grace)"
)
print(" ".join(cmd))

session_start = time.monotonic()
with open(runner_log_path, "w") as log_fh:
    proc = subprocess.Popen(
        cmd, cwd=REPO_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        print(line, end="")
        log_fh.write(line)
    exit_code = proc.wait()
session_elapsed_min = (time.monotonic() - session_start) / 60.0

gpu_sampler.terminate()
try:
    gpu_sampler.wait(timeout=5)
except subprocess.TimeoutExpired:
    gpu_sampler.kill()

EXIT_MEANINGS = {
    0: "clean stop (drained or SIGTERM-checkpointed) ",
    3: "queue fully drained — nothing left to run",
    4: "canary preflight failed inside the runner (only reachable with EVOLVE=True)",
}
print(
    f"\nrunner exited {exit_code}: {EXIT_MEANINGS.get(exit_code, 'unexpected exit — check the log above')}"
)
print(f"elapsed: {session_elapsed_min:.1f} min")
check("runner exit code is a known-good code (0 or 3)", exit_code in (0, 3), f"exit={exit_code}")

## Checklist 5 — Verify the session actually made progress

A process that ran is not the same as progress: confirm the queue moved *and* that any experiment this session touched left behind a checkpoint newer than the session start, not just a claim that got orphaned.

In [ ]:
snapshot_after = queue.snapshot()
before_by_name = {s.name: s for s in snapshot_before}
changed = [
    (s.name, before_by_name[s.name].status.value, s.status.value)
    for s in snapshot_after
    if s.name in before_by_name and s.status != before_by_name[s.name].status
]

print(f"{'name':<28}{'before':<10}{'after':<10}")
for name, before, after in changed:
    print(f"{name:<28}{before:<10}{after:<10}")

check("queue status changed for at least one entry", bool(changed), f"{len(changed)} changed")

newly_done = [n for n, before, after in changed if after == "done"]
print(f"newly done this session: {newly_done or 'none'}")

In [ ]:
from training.checkpoint import CheckpointManager  # noqa: E402

touched_names = {n for n, _, _ in changed} | {
    s.name for s in snapshot_after if s.status is ExperimentStatus.RUNNING
}
session_start_wall = datetime.now(timezone.utc).timestamp() - session_elapsed_min * 60

stale_checkpoints = []
for name in sorted(touched_names):
    manager = CheckpointManager(profile.paths.ckpt_root / name)
    latest = manager.latest()
    if latest is None:
        stale_checkpoints.append((name, "no checkpoint directory at all"))
        continue
    mtime = latest.stat().st_mtime
    if mtime < session_start_wall - 60:  # 60s slack for clock skew
        stale_checkpoints.append((name, f"latest checkpoint predates this session ({latest.name})"))

check(
    "every touched experiment left a fresh checkpoint",
    not stale_checkpoints,
    "; ".join(f"{n}: {reason}" for n, reason in stale_checkpoints) or "all fresh",
)

## Checklist 6 — GPU utilization sanity

In [ ]:
import csv

util_samples = []
if gpu_log_path.exists():
    with open(gpu_log_path) as fh:
        for row in csv.reader(fh):
            if len(row) >= 2:
                try:
                    util_samples.append(float(row[1].strip().rstrip(" %")))
                except ValueError:
                    continue

if util_samples:
    avg_util = sum(util_samples) / len(util_samples)
    max_util = max(util_samples)
    print(f"GPU util over {len(util_samples)} samples: avg={avg_util:.1f}%  max={max_util:.1f}%")
    check(
        f"avg GPU util >= {LOW_GPU_UTIL_WARN_PCT:.0f}%",
        avg_util >= LOW_GPU_UTIL_WARN_PCT,
        f"avg={avg_util:.1f}%",
    )
else:
    print("no GPU samples captured (session shorter than the 60s sampling interval?)")
    check("GPU utilization sampled", False, "no samples")

## Checklist 7 — Session handoff log

Appends a record so the *next* session — possibly launched hours later with no shared chat context — knows exactly what state the queue and window budget are in.

In [ ]:
state = {"window_started_utc": None, "window_hours": WINDOW_HOURS, "sessions": []}
if SESSION_STATE_PATH.exists():
    state = json.loads(SESSION_STATE_PATH.read_text())

if state["window_started_utc"] is None:
    state["window_started_utc"] = datetime.now(timezone.utc).isoformat()

session_record = {
    "session_id": session_id,
    "started_utc": datetime.fromtimestamp(session_start_wall, tz=timezone.utc).isoformat(),
    "elapsed_minutes": round(session_elapsed_min, 1),
    "exit_code": exit_code,
    "queue_changed": changed,
    "newly_done": newly_done,
    "stale_checkpoints": stale_checkpoints,
    "gpu_avg_util_pct": round(avg_util, 1) if util_samples else None,
    "checklist": [{"label": label, "ok": ok, "detail": detail} for label, ok, detail in checklist],
}
state["sessions"].append(session_record)
SESSION_STATE_PATH.write_text(json.dumps(state, indent=2))

window_started = datetime.fromisoformat(state["window_started_utc"])
window_elapsed_h = (datetime.now(timezone.utc) - window_started).total_seconds() / 3600.0
window_remaining_h = WINDOW_HOURS - window_elapsed_h

with open(SESSION_LOG_PATH, "a") as fh:
    fh.write(f"## Session {session_id}\n")
    fh.write(f"- elapsed: {session_elapsed_min:.1f} min, exit={exit_code}\n")
    fh.write(f"- queue changes: {changed or 'none'}\n")
    fh.write(f"- newly done: {newly_done or 'none'}\n")
    fh.write(f"- stale checkpoints: {stale_checkpoints or 'none'}\n")
    fh.write(f"- gpu avg util: {round(avg_util, 1) if util_samples else 'n/a'}%\n")
    fh.write(f"- window remaining: {window_remaining_h:.1f}h of {WINDOW_HOURS}h\n\n")

print(f"session log: {SESSION_LOG_PATH}")
print(
    f"window remaining: {window_remaining_h:.1f}h of {WINDOW_HOURS}h ({len(state['sessions'])} sessions run)"
)

## Final checklist summary

In [ ]:
print("=" * 60)
print(f"SESSION {session_id} SUMMARY")
print("=" * 60)
for label, ok, detail in checklist:
    print(f"[{'PASS' if ok else 'FAIL'}] {label}" + (f" — {detail}" if detail else ""))

all_ok = all(ok for _, ok, _ in checklist)
queue_drained = exit_code == 3

print()
if not all_ok:
    print(
        "ACTION: something above FAILed — investigate before the next session; "
        "do not assume unattended progress happened."
    )
elif queue_drained:
    print(
        "ACTION: queue is fully drained. Nothing left to run — stop here, or set "
        "EVOLVE=True / repopulate experiments/queue.yaml (training.sweeps) for more work."
    )
elif window_remaining_h <= 0:
    print("ACTION: allocation window is used up. Stop; resume in the next window.")
else:
    print(
        f"ACTION: all checks passed, {window_remaining_h:.1f}h remain in the window — "
        "re-run this notebook top-to-bottom for the next session."
    )